### Generate droplet trajectories


In [ ]:
import yt
import os
import numpy as np
import glob
import re

# Parameters
alpha0 = 1.5
temperature = 0.0
nx, ny, nz = 32, 32, 32
radius = 0.2
rho_lo = 0.0
rho_hi = 3.0
step1 = 360000
step2 = 600000
plot_int = 1000

# Construct file paths
data_dir = "./data_droplet_alpha0_{:.2f}_r{:.2f}_size{:d}-{:d}-{:d}/".format(alpha0, radius, nx, ny, nz)

IMG_folder = data_dir + "/IMG_data_shshan_alpha0_{:.2f}_xi_{:.1e}_size{:d}-{:d}-{:d}/".format(alpha0, temperature, nx, ny, nz)

# Create image directory if it doesn't exist
os.makedirs(IMG_folder, exist_ok=True)
all_plt_files = sorted(glob.glob(os.path.join(data_dir, "plt*")))

# Extract step numbers from filenames (assuming format like plt00010, plt12345, etc.)
def extract_step(filepath):
    basename = os.path.basename(filepath) # Get the file/folder name from the path
    match = re.search(r'plt(\d+)', basename)  # It finds single digits with \d and sequences of digits with \d+.
    if match:
        return int(match.group(1)) # Extract the digits in filename and convert to integer
    else:
        return -1  # invalid
# Load all plot files (e.g., plt0000, plt0001, ...)
ts = yt.load(data_dir + "plt*")

# Filter files within [step1, step2] and divisible by plot_int (or aligned to step1 + n*plot_int)
selected_files = []
for f in all_plt_files:
    step = extract_step(f)
    if step == -1:
        continue
    if step1 <= step <= step2 and (step - step1) % plot_int == 0:
        selected_files.append((step, f))

# Sort by step number
selected_files.sort(key=lambda x: x[0])

# Process each selected file
ac = 0
for step, filepath in selected_files:
    fr = yt.load(filepath)
    slc = yt.SlicePlot(fr, "z", ("boxlib", "rhoA"))
    slc.set_log(("boxlib", "rhoA"), False)
    # Optional: set color limits
    # slc.set_zlim(("boxlib", "rho"), rho_lo - 0.3, rho_hi + 0.5)
    
    filename = os.path.join(IMG_folder, f"fr{ac:07d}.png")
    slc.save(filename)
    ac += 1

print(f"Saved {ac} frames from step {step1} to {step2} (interval={plot_int})")